In [ ]:
"""
Author: Aaron Niecestro
Project: Course Recommender System Capstone

Created: November 3, 2026
Last Edit: November 9, 2026
Progress: Complete

Analysis: 02 - Clustering-Based Recommender System
Method: K-Means Clustering + Principal Component Analysis (PCA)

Description:

Groups users according to their learning interests and
generates personalized course recommendations based on
popular courses within each cluster.

User profile features are standardized before applying
K-Means clustering. Silhouette analysis is used to
determine the optimal number of clusters, while PCA is
used to reduce dimensionality and improve visualization
and model efficiency.

Techniques Applied:

- Feature Standardization
- K-Means Clustering
- Silhouette Analysis
- Principal Component Analysis (PCA)
- Cluster-Based Recommendation Engine

Datasets:

- user_profile.csv
- ratings.csv
- course_genre.csv

Key Parameters:

- Silhouette-Based Cluster Selection
- PCA Variance Threshold = 90%
- Popularity Threshold = 10 Enrollments

Research Questions:

- Can users be grouped based on learning interests?
- What is the optimal number of clusters?
- How much dimensionality reduction can PCA achieve?
- Can cluster enrollments be used to generate meaningful
  course recommendations?

Outputs:

- User cluster assignments
- Silhouette analysis results
- PCA-transformed features
- Cluster recommendation results
- Recommendation evaluation metrics
"""

============================================================
Clustering-Based Course Recommender System
IBM Machine Learning Capstone

Author: Aaron Niecestro

Objective:
- Cluster users based on learning interests
- Reduce dimensions with PCA
- Generate cluster-based recommendations
- Analyze recommendation performance
============================================================

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
from pathlib import Path

============================================================
Reproducibility
============================================================

In [ ]:
import random

In [ ]:
SEED = 423

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
RANDOM_STATE = SEED

============================================================
Configuration
============================================================

In [ ]:
MAX_CLUSTERS = 15
PCA_VARIANCE_THRESHOLD = 0.90
POPULARITY_THRESHOLD = 10

In [ ]:
sns.set_style("whitegrid")

============================================================
Paths
============================================================

In [ ]:
DATA_DIR = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\capstone\00-Data"
)

In [ ]:
OUTPUT_DIR = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\capstone\02-Unsupervised\unsupervised-results"
)

In [ ]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

============================================================
Runtime Tracking
============================================================

In [ ]:
import time

In [ ]:
SCRIPT_START = time.time()

============================================================
Load Data
============================================================

In [ ]:
print("\nLoading Data...")
print("-" * 40)

In [ ]:
user_profile_df = pd.read_csv(
    DATA_DIR / "user_profile.csv"
)

In [ ]:
ratings_df = pd.read_csv(
    DATA_DIR / "ratings.csv"
)

In [ ]:
course_genres_df = pd.read_csv(
    DATA_DIR / "course_genre.csv"
)

In [ ]:
print("Data Loaded Successfully")

In [ ]:
print(
    f"User Profiles Shape : "
    f"{user_profile_df.shape}"
)

In [ ]:
print(
    f"Ratings Shape       : "
    f"{ratings_df.shape}"
)

In [ ]:
print(
    f"Course Genres Shape : "
    f"{course_genres_df.shape}"
)

============================================================
Prepare Features
============================================================

In [ ]:
print("\nPreparing Features...")
print("-" * 40)

In [ ]:
feature_names = list(user_profile_df.columns[1:])

In [ ]:
scaler = StandardScaler()

In [ ]:
user_profile_df[feature_names] = scaler.fit_transform(
    user_profile_df[feature_names]
)

In [ ]:
features = user_profile_df.drop(
    columns=["user"]
)

In [ ]:
user_ids = user_profile_df[["user"]]

In [ ]:
print(
    f"Total Users    : {len(user_profile_df):,}"
)

In [ ]:
print(
    f"Total Features : {len(feature_names)}"
)

In [ ]:
print("Feature Preparation Complete")

============================================================
K-Means Optimization
============================================================

In [ ]:
print("\nOptimizing K-Means Clusters...")
print("-" * 40)

In [ ]:
k_values = range(2, MAX_CLUSTERS + 1)

In [ ]:
inertias = []
silhouette_scores = []

In [ ]:
for k in k_values:

    print(f"\nEvaluating K = {k}")

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )

    labels = model.fit_predict(features)

    inertias.append(model.inertia_)

    sample_idx = (
        features.sample(
            min(5000, len(features)),
            random_state=RANDOM_STATE
        )
        .index
    )

    score = silhouette_score(
        features.loc[sample_idx],
        labels[sample_idx]
    )

    silhouette_scores.append(score)

    print(
        f"   Inertia = {model.inertia_:,.2f} | "
        f"Silhouette = {score:.4f}"
    )

In [ ]:
best_k = k_values[
    np.argmax(silhouette_scores)
]

In [ ]:
print("\n" + "-" * 40)
print(f"Optimal Clusters Found : {best_k}")
print(
    f"Best Silhouette Score  : "
    f"{max(silhouette_scores):.4f}"
)
print("K-Means Optimization Complete")

============================================================
Elbow + Silhouette Plot
============================================================

In [ ]:
print("\nGenerating K-Means Evaluation Plots...")
print("-" * 40)

In [ ]:
fig, ax = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

------------------------------------------------------------
Elbow Plot
------------------------------------------------------------

In [ ]:
ax[0].plot(
    k_values,
    inertias,
    marker="o"
)

In [ ]:
ax[0].axvline(
    x=best_k,
    color="red",
    linestyle="--",
    label=f"Best K = {best_k}"
)

In [ ]:
ax[0].scatter(
    best_k,
    inertias[best_k - 2],
    color="red",
    s=100,
    zorder=5
)

In [ ]:
ax[0].set_title(
    "Elbow Method"
)

In [ ]:
ax[0].set_xlabel(
    "Number of Clusters (K)"
)

In [ ]:
ax[0].set_ylabel(
    "Inertia"
)

In [ ]:
ax[0].legend()

------------------------------------------------------------
Silhouette Plot
------------------------------------------------------------

In [ ]:
ax[1].plot(
    k_values,
    silhouette_scores,
    marker="o",
    color="green"
)

In [ ]:
ax[1].axvline(
    x=best_k,
    color="red",
    linestyle="--",
    label=f"Best K = {best_k}"
)

In [ ]:
ax[1].scatter(
    best_k,
    max(silhouette_scores),
    color="red",
    s=100,
    zorder=5
)

In [ ]:
ax[1].set_title(
    "Silhouette Scores"
)

In [ ]:
ax[1].set_xlabel(
    "Number of Clusters (K)"
)

In [ ]:
ax[1].set_ylabel(
    "Silhouette Score"
)

In [ ]:
ax[1].legend()

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
print("K-Means Evaluation Plots Complete")

============================================================
Train Final K-Means Model
============================================================

In [ ]:
print("\nTraining Final K-Means Model...")
print("-" * 40)

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=10
)

In [ ]:
cluster_labels = kmeans.fit_predict(
    features
)

In [ ]:
cluster_df = pd.DataFrame({
    "user": user_ids["user"],
    "cluster": cluster_labels
})

In [ ]:
print(
    f"Users Assigned to Clusters : "
    f"{len(cluster_df):,}"
)

In [ ]:
print(
    f"Clusters Created           : "
    f"{cluster_df['cluster'].nunique()}"
)

In [ ]:
cluster_sizes = (
    cluster_df["cluster"]
    .value_counts()
    .sort_index()
)

In [ ]:
print("\nCluster Sizes")
print("-" * 40)

In [ ]:
for cluster_id, count in cluster_sizes.items():

    print(
        f"Cluster {cluster_id:<2}: "
        f"{count:>6,} users"
    )

In [ ]:
print("\nCluster Assignment Sample")
print(cluster_df.head())

In [ ]:
print("Final K-Means Model Complete")

============================================================
PCA Optimization
============================================================

In [ ]:
print("\nRunning PCA Optimization...")
print("-" * 40)

In [ ]:
variance_explained = []

In [ ]:
for n in range(
    1,
    features.shape[1] + 1
):

    pca = PCA(
        n_components=n
    )

    pca.fit(features)

    variance_explained.append(
        np.sum(
            pca.explained_variance_ratio_
        )
    )

In [ ]:
best_components = next(
    i + 1
    for i, var in enumerate(
        variance_explained
    )
    if var >= PCA_VARIANCE_THRESHOLD
)

In [ ]:
print(
    f"Original Features          : "
    f"{features.shape[1]}"
)

In [ ]:
print(
    f"Selected Components        : "
    f"{best_components}"
)

In [ ]:
print(
    f"Cumulative Variance Kept   : "
    f"{variance_explained[best_components - 1]:.2%}"
)

In [ ]:
print("PCA Optimization Complete")

============================================================
PCA Variance Plot
============================================================

In [ ]:
print("\nGenerating PCA Variance Plot...")
print("-" * 40)

In [ ]:
plt.figure(
    figsize=(8, 5)
)

In [ ]:
plt.plot(
    range(
        1,
        features.shape[1] + 1
    ),
    variance_explained,
    marker="o"
)

90% variance threshold

In [ ]:
plt.axhline(
    y=PCA_VARIANCE_THRESHOLD,
    color="red",
    linestyle="--",
    label="90% Variance Threshold"
)

Selected component count

In [ ]:
plt.axvline(
    x=best_components,
    color="green",
    linestyle="--",
    label=f"Selected Components = {best_components}"
)

In [ ]:
plt.scatter(
    best_components,
    variance_explained[best_components - 1],
    color="green",
    s=100,
    zorder=5
)

In [ ]:
plt.xlabel(
    "Number of Components"
)

In [ ]:
plt.ylabel(
    "Cumulative Explained Variance"
)

In [ ]:
plt.title(
    "PCA Optimization"
)

In [ ]:
plt.legend()

In [ ]:
plt.tight_layout()

In [ ]:
plt.show()

In [ ]:
print(
    f"Variance Explained : "
    f"{variance_explained[best_components - 1]:.2%}"
)

In [ ]:
print("PCA Variance Plot Complete")

============================================================
PCA Transformation
============================================================

In [ ]:
print("\nTransforming Features with PCA...")
print("-" * 40)

In [ ]:
pca = PCA(
    n_components=best_components
)

In [ ]:
pca_components = pca.fit_transform(
    features
)

In [ ]:
pca_df = pd.DataFrame(
    pca_components
)

In [ ]:
pca_df.insert(
    0,
    "user",
    user_ids["user"]
)

In [ ]:
print(
    f"Original Dimensions  : "
    f"{features.shape[1]}"
)

In [ ]:
print(
    f"PCA Components       : "
    f"{best_components}"
)

In [ ]:
print(
    f"Dimensions Reduced   : "
    f"{features.shape[1] - best_components}"
)

In [ ]:
print(
    f"Reduction Percentage : "
    f"{((features.shape[1] - best_components) / features.shape[1]) * 100:.2f}%"
)

In [ ]:
print("PCA Transformation Complete")

============================================================
PCA-Based K-Means
============================================================

In [ ]:
print("\nRunning K-Means on PCA Components...")
print("-" * 40)

In [ ]:
kmeans_pca = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=10
)

In [ ]:
pca_cluster_labels = kmeans_pca.fit_predict(
    pca_components
)

In [ ]:
pca_silhouette = silhouette_score(
    pca_components,
    pca_cluster_labels
)

In [ ]:
print(
    f"PCA Silhouette Score : "
    f"{pca_silhouette:.4f}"
)

In [ ]:
pca_cluster_df = pd.DataFrame({
    "user": user_ids["user"],
    "cluster": pca_cluster_labels
})

In [ ]:
print(
    f"Users Assigned to Clusters : "
    f"{len(pca_cluster_df):,}"
)

In [ ]:
print(
    f"Clusters Created           : "
    f"{pca_cluster_df['cluster'].nunique()}"
)

In [ ]:
pca_cluster_sizes = (
    pca_cluster_df["cluster"]
    .value_counts()
    .sort_index()
)

In [ ]:
print("\nPCA Cluster Sizes")
print("-" * 40)

In [ ]:
for cluster_id, count in pca_cluster_sizes.items():

    print(
        f"Cluster {cluster_id:<2}: "
        f"{count:>6,} users"
    )

In [ ]:
print("\nPCA Cluster Assignment Sample")
print(
    pca_cluster_df.head()
)

In [ ]:
print("PCA-Based K-Means Complete")

============================================================
Cluster-Based Recommendation Engine
============================================================

In [ ]:
print("\nGenerating Recommendations...")
print("-" * 40)

In [ ]:
test_users_df = ratings_df[
    ["user", "item"]
]

In [ ]:
test_users_labelled = pd.merge(
    test_users_df,
    pca_cluster_df,
    on="user"
)

In [ ]:
course_counts = (
    test_users_labelled
    .groupby(
        ["cluster", "item"]
    )
    .size()
    .reset_index(
        name="enrollments"
    )
)

In [ ]:
recommended_users = []
recommended_courses = []
recommendation_scores = []

In [ ]:
unique_users = (
    test_users_labelled["user"]
    .unique()
)

In [ ]:
print(
    f"Users Evaluated : "
    f"{len(unique_users):,}"
)

In [ ]:
for user_id in unique_users:

    user_subset = (
        test_users_labelled[
            test_users_labelled["user"]
            == user_id
        ]
    )

    enrolled_courses = set(
        user_subset["item"]
    )

    cluster_id = (
        user_subset["cluster"]
        .iloc[0]
    )

    popular_courses = (
        course_counts[
            (
                course_counts["cluster"]
                == cluster_id
            )
            &
            (
                course_counts[
                    "enrollments"
                ]
                >= POPULARITY_THRESHOLD
            )
        ]
    )

    for _, row in popular_courses.iterrows():

        course = row["item"]
        count = row["enrollments"]

        if course not in enrolled_courses:

            recommended_users.append(
                user_id
            )

            recommended_courses.append(
                course
            )

            recommendation_scores.append(
                count
            )

In [ ]:
recommendation_df = pd.DataFrame({

    "USER":
    recommended_users,

    "COURSE_ID":
    recommended_courses,

    "SCORE":
    recommendation_scores

In [ ]:
})

In [ ]:
print(
    f"Recommendations Generated : "
    f"{len(recommendation_df):,}"
)

In [ ]:
print(
    f"Unique Courses Recommended : "
    f"{recommendation_df['COURSE_ID'].nunique():,}"
)

In [ ]:
print(
    f"Users Receiving Recommendations : "
    f"{recommendation_df['USER'].nunique():,}"
)

In [ ]:
print("Recommendation Generation Complete")

============================================================
Analytics
============================================================

In [ ]:
print("\nRunning Recommendation Analytics...")
print("-" * 40)

In [ ]:
avg_recommendations = (
    recommendation_df
    .groupby("USER")
    .size()
    .mean()
)

In [ ]:
top_courses = (
    recommendation_df["COURSE_ID"]
    .value_counts()
    .head(10)
)

In [ ]:
course_lookup = dict(
    zip(
        course_genres_df["COURSE_ID"],
        course_genres_df["TITLE"]
    )
)

In [ ]:
print(
    f"Users Receiving Recommendations : "
    f"{recommendation_df['USER'].nunique():,}"
)

In [ ]:
print(
    f"Unique Courses Recommended      : "
    f"{recommendation_df['COURSE_ID'].nunique():,}"
)

In [ ]:
print(
    f"Total Recommendations Generated : "
    f"{len(recommendation_df):,}"
)

In [ ]:
print(
    f"Average Recommendations/User    : "
    f"{avg_recommendations:.2f}"
)

In [ ]:
print(
    f"Average Recommendation Score    : "
    f"{recommendation_df['SCORE'].mean():.2f}"
)

In [ ]:
print("\nTop 10 Recommended Courses")
print("-" * 100)

In [ ]:
for course_id, count in top_courses.items():

    course_title = course_lookup.get(
        course_id,
        "Unknown Course"
    )

    print(
        f"{course_id:<12} "
        f"{count:>8,}   "
        f"{course_title}"
    )

In [ ]:
print("\nRecommendation Analytics Complete")

============================================================
Visualization
============================================================

In [ ]:
print("\nGenerating Visualizations...")
print("-" * 40)

In [ ]:
plot_df = pd.DataFrame({
    "Course_ID": top_courses.index,
    "Count": top_courses.values
})

In [ ]:
plot_df["Course"] = (
    plot_df["Course_ID"]
    .map(course_lookup)
)

In [ ]:
colors = ["#1f77b4"] + ["#7daed3"] * 9

In [ ]:
plt.figure(figsize=(12, 6))

In [ ]:
ax = sns.barplot(
    data=plot_df,
    x="Count",
    y="Course",
    palette=colors
)

In [ ]:
# Add data labels
for i, value in enumerate(plot_df["Count"]):
    ax.text(
        value + 25,
        i,
        f"{value:,}",
        va="center"
    )

In [ ]:
# Give labels room on right side
plt.xlim(
    0,
    plot_df["Count"].max() * 1.06
)

In [ ]:
plt.title(
    "Top 10 Recommended Courses",
    fontsize=14,
    fontweight="bold"
)

In [ ]:
plt.xlabel(
    "Recommendation Count"
)

In [ ]:
plt.ylabel(
    "Course"
)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR / "top_recommended_courses.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.show()

In [ ]:
print(
    f"Courses Displayed        : "
    f"{len(top_courses)}"
)

In [ ]:
print(
    f"Most Recommended Course  : "
    f"{top_courses.index[0]}"
)

In [ ]:
print(
    f"Recommendation Count     : "
    f"{top_courses.iloc[0]:,}"
)

In [ ]:
print("Visualization Complete")

============================================================
Export Results
============================================================

In [ ]:
print("\nSaving Results...")
print("-" * 40)

In [ ]:
cluster_df.to_csv(
    OUTPUT_DIR / "user_clusters.csv",
    index=False
)

In [ ]:
pca_df.to_csv(
    OUTPUT_DIR / "pca_features.csv",
    index=False
)

In [ ]:
recommendation_df.to_csv(
    OUTPUT_DIR / "cluster_recommendations.csv",
    index=False
)

In [ ]:
print(
    f"Cluster Assignments Saved : "
    f"{len(cluster_df):,} records"
)

In [ ]:
print(
    f"PCA Features Saved        : "
    f"{len(pca_df):,} records"
)

In [ ]:
print(
    f"Recommendations Saved     : "
    f"{len(recommendation_df):,} records"
)

In [ ]:
print(
    f"\nResults saved to:\n{OUTPUT_DIR}"
)

In [ ]:
print("Export Complete")

============================================================
Final Summary
============================================================

In [ ]:
total_runtime = time.time() - SCRIPT_START

In [ ]:
print("\n==============================")
print("FINAL SUMMARY")
print("==============================")

In [ ]:
print(
    f"Users Clustered             : "
    f"{len(user_profile_df):,}"
)

In [ ]:
print(
    f"Optimal Clusters            : "
    f"{best_k}"
)

In [ ]:
print(
    f"PCA Components Retained     : "
    f"{best_components}"
)

In [ ]:
print(
    f"Recommendations Generated   : "
    f"{len(recommendation_df):,}"
)

In [ ]:
print(
    f"Unique Courses Recommended  : "
    f"{recommendation_df['COURSE_ID'].nunique():,}"
)

In [ ]:
print(
    f"Average Recommendations/User: "
    f"{avg_recommendations:.2f}"
)

In [ ]:
print(
    f"Average Recommendation Score: "
    f"{recommendation_df['SCORE'].mean():.2f}"
)

In [ ]:
print(
    f"Total Runtime               : "
    f"{total_runtime / 60:.2f} minutes"
)

In [ ]:
print("\nPipeline Complete")